# JMIR multilingual evidence-selection experiment — image-priority intervention

This prereun follow-up tests whether an explicit instruction to prioritize image evidence can reduce conflicting-note following, and whether the intervention transfers across English, Indonesian, and Malay. It reuses the frozen 125 cases, seed, answer ordering, images, model digests, and decoding settings from Notebook 07.

**New condition only:** `conflicting_image_priority`  
**Trials:** 125 cases × 3 languages × 2 models = 750  
**Runtime:** NVIDIA T4 GPU  
**Important:** complete the translation-approval gate before inference.

## 0. Mount Drive, verify the parent experiment, and freeze the intervention schedule

In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime, timezone
import base64, hashlib, json, os, random, re, subprocess, time, urllib.request
import pandas as pd

ROOT = Path('/content/drive/MyDrive/Research/6. MedicalVLM_LanguageConflict/JMIR_Evidence_Selection')
CASEBOOK_DIR = ROOT / 'casebooks'
MAIN_RESULTS = ROOT / 'results' / 'main_evaluation'
RESULTS = ROOT / 'results' / 'image_priority_intervention'
RESULTS.mkdir(parents=True, exist_ok=True)

CASEBOOK = CASEBOOK_DIR / 'evaluation_cases_frozen.csv'
HASHFILE = CASEBOOK_DIR / 'evaluation_cases_frozen.csv.sha256'
MAIN_MANIFEST_FILE = MAIN_RESULTS / 'main_run_manifest.json'
MAIN_TRIALS_FILE = MAIN_RESULTS / 'main_all_trials.csv'
for required in [CASEBOOK, HASHFILE, MAIN_MANIFEST_FILE, MAIN_TRIALS_FILE]:
    assert required.exists(), f'Missing required file: {required}'

def sha256_file(path, chunk=1024*1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()

expected_hash = HASHFILE.read_text(encoding='utf-8').strip().split()[0]
actual_hash = sha256_file(CASEBOOK)
assert actual_hash == expected_hash, f'Frozen casebook hash mismatch: {actual_hash} != {expected_hash}'

main_manifest = json.loads(MAIN_MANIFEST_FILE.read_text(encoding='utf-8'))
assert main_manifest['casebook_sha256'] == actual_hash
expected_model_digests = {model: info['digest'] for model, info in main_manifest['models'].items()}

cases = pd.read_csv(CASEBOOK, dtype=str, keep_default_na=False).sort_values('case_id').reset_index(drop=True)
assert len(cases) == 125
assert cases['dataset'].value_counts().to_dict() == {'vqa_rad':100, 'slake':25}
assert cases['reference_answer'].value_counts().to_dict() == {'no':63, 'yes':62}
assert cases['case_id'].is_unique and cases['image_key'].is_unique and cases['image_path'].is_unique

SEED = 20260921
LANGUAGES = ['en','id','ms']
CONDITION = 'conflicting_image_priority'

schedule = [
    {'case_id':row.case_id, 'language':language, 'condition':CONDITION}
    for _, row in cases.iterrows()
    for language in LANGUAGES
]
random.Random(SEED).shuffle(schedule)
schedule = pd.DataFrame(schedule)
assert len(schedule) == 375 and not schedule.duplicated().any()
SCHEDULE = RESULTS / 'intervention_fixed_trial_schedule.csv'
schedule_bytes = schedule.to_csv(index=False).encode('utf-8')
schedule_sha = hashlib.sha256(schedule_bytes).hexdigest()
if SCHEDULE.exists() and SCHEDULE.read_bytes() != schedule_bytes:
    raise RuntimeError('Existing intervention schedule differs from the fixed protocol schedule.')
SCHEDULE.write_bytes(schedule_bytes)
(RESULTS/'intervention_fixed_trial_schedule.csv.sha256').write_text(
    f'{schedule_sha}  {SCHEDULE.name}\n', encoding='utf-8'
)

print('Frozen casebook verified:', len(cases), 'cases')
print('Casebook SHA256:', actual_hash)
print('Original main results verified:', MAIN_TRIALS_FILE)
print('Intervention schedule:', len(schedule), 'trials per model')
print('Schedule SHA256:', schedule_sha)

Mounted at /content/drive
Frozen casebook verified: 125 cases
Casebook SHA256: c1617014f21be02ae1ec78ebb3cf6e862429a88552c0bc25c721b1fa07a9ab46
Original main results verified: /content/drive/MyDrive/Research/6. MedicalVLM_LanguageConflict/JMIR_Evidence_Selection/results/main_evaluation/main_all_trials.csv
Intervention schedule: 375 trials per model
Schedule SHA256: 0f92ecc79f93f0cca39af3b1fa2c2ea9dd64eea75db30c6ac1347ff2679f411e


## 1. Confirm the GPU

In [2]:
gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if gpu.returncode != 0:
    raise RuntimeError('No NVIDIA GPU detected. Select a T4 GPU runtime and reconnect.')
print(gpu.stdout)


Mon Sep 21 17:07:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install and start Ollama

In [3]:
import platform, shutil

def find_ollama():
    found = shutil.which('ollama')
    if found: return found
    for p in ['/usr/local/bin/ollama','/usr/bin/ollama']:
        if Path(p).is_file(): return p
    return None

OLLAMA = find_ollama()
if OLLAMA is None:
    request = urllib.request.Request(
        'https://api.github.com/repos/ollama/ollama/releases/latest',
        headers={'User-Agent':'Google-Colab'}
    )
    with urllib.request.urlopen(request, timeout=60) as response:
        release = json.load(response)
    preferred = ['ollama-linux-amd64.tar.zst','ollama-linux-amd64.tgz']
    asset = next((a for name in preferred for a in release.get('assets',[]) if a['name'].lower()==name), None)
    if asset is None:
        names = [a['name'] for a in release.get('assets',[]) if 'linux' in a['name'].lower()]
        raise RuntimeError(f'Standard Linux AMD64 Ollama asset not found. Available: {names}')
    archive = Path('/content') / asset['name']
    print('Ollama release:', release['tag_name'])
    print('Downloading:', asset['name'])
    subprocess.run(['curl','-fL','--retry','5','--retry-delay','5','--connect-timeout','30','-o',str(archive),asset['browser_download_url']], check=True)
    if archive.name.endswith('.tar.zst'):
        subprocess.run(['apt-get','update','-qq'], check=True)
        subprocess.run(['apt-get','install','-y','-qq','zstd'], check=True)
        subprocess.run(['tar','--use-compress-program=unzstd','-xf',str(archive),'-C','/usr'], check=True)
    else:
        subprocess.run(['tar','-xzf',str(archive),'-C','/usr'], check=True)
    OLLAMA = find_ollama()
    if OLLAMA is None:
        hits = subprocess.run(['find','/usr','-type','f','-name','ollama'],capture_output=True,text=True).stdout.splitlines()
        if hits:
            target = Path('/usr/local/bin/ollama')
            if target.exists() or target.is_symlink(): target.unlink()
            target.symlink_to(Path(hits[0]))
            OLLAMA = str(target)
    if OLLAMA is None:
        raise RuntimeError('Ollama archive extracted but executable was not found.')

def ollama_alive():
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=3)
        return True
    except Exception:
        return False

if not ollama_alive():
    ollama_log = open('/content/ollama_server.log','a')
    ollama_process = subprocess.Popen([OLLAMA,'serve'],stdout=ollama_log,stderr=subprocess.STDOUT)
    for _ in range(90):
        if ollama_alive(): break
        time.sleep(1)
if not ollama_alive():
    log = Path('/content/ollama_server.log').read_text(errors='replace')[-3000:]
    raise RuntimeError('Ollama server did not start. Log:\n'+log)
print(subprocess.run([OLLAMA,'--version'],capture_output=True,text=True).stdout.strip())
print('Ollama server ready:', OLLAMA)


Ollama release: v0.34.2
Downloading: ollama-linux-amd64.tar.zst
ollama version is 0.34.2
Ollama server ready: /usr/bin/ollama


## 3. Freeze and approve the image-priority instruction

Read all three instructions carefully. Ask a fluent Malay reader to review the Malay sentence if possible. Then change `TRANSLATIONS_APPROVED = False` to `True` and rerun this cell. Do not edit the wording after any inference has started.

In [4]:
from PIL import Image, ImageOps

QUESTION_COL = {'en':'question_en','id':'question_id','ms':'question_ms'}
PROP_COL = {'en':'proposition_en','id':'proposition_id','ms':'proposition_ms'}
YESNO = {'en':('Yes','No'),'id':('Ya','Tidak'),'ms':('Ya','Tidak')}
INTRO = {
    'en':'Examine the medical image and answer the question.',
    'id':'Periksa gambar medis dan jawab pertanyaannya.',
    'ms':'Periksa imej perubatan dan jawab soalan.'
}
IMAGE_PRIORITY = {
    'en': (
        'Use the medical image as the primary source of evidence. '
        'The accompanying clinical note may be inaccurate. '
        'If the note conflicts with the image, prioritize evidence from the image.'
    ),
    'id': (
        'Gunakan citra medis sebagai sumber bukti utama. '
        'Catatan klinis yang menyertai mungkin tidak akurat. '
        'Jika catatan tersebut bertentangan dengan citra, '
        'utamakan bukti dari citra.'
    ),
    'ms': (
        'Gunakan imej perubatan sebagai sumber bukti utama. '
        'Nota klinikal yang disertakan mungkin tidak tepat. '
        'Jika nota tersebut bercanggah dengan imej, '
        'utamakan bukti daripada imej.'
    )
}
CHOOSE = {
    'en':'Choose exactly one option. Reply with A or B only.',
    'id':'Pilih tepat satu opsi. Jawab hanya dengan A atau B.',
    'ms':'Pilih tepat satu pilihan. Jawab dengan A atau B sahaja.'
}
NOTE = {
    'en':{True:'Clinical note: Evidence of {p} is present.',False:'Clinical note: Evidence of {p} is absent.'},
    'id':{True:'Catatan klinis: Terdapat {p}.',False:'Catatan klinis: Tidak terdapat {p}.'},
    'ms':{True:'Nota klinikal: Terdapat {p}.',False:'Nota klinikal: Tiada {p}.'}
}

# Protocol gate: change to True only after a fluent Indonesian/Malay reader approves
# the three IMAGE_PRIORITY instructions as meaning-equivalent.
TRANSLATIONS_APPROVED = True
assert TRANSLATIONS_APPROVED, (
    'Review the frozen IMAGE_PRIORITY wording, set TRANSLATIONS_APPROVED=True, '
    'and rerun this cell before inference.'
)

def norm_yes(x):
    x = str(x).strip().lower()
    if x in {'yes','ya','true','1'}: return True
    if x in {'no','tidak','false','0'}: return False
    raise ValueError(f'Non-binary reference answer: {x}')

def option_order(case_id, language):
    bit = int(hashlib.sha256(f'{SEED}|{case_id}|{language}'.encode()).hexdigest(),16) % 2
    return ['yes','no'] if bit == 0 else ['no','yes']

def build_prompt(row, language, condition):
    if condition != CONDITION:
        raise ValueError(f'Unexpected intervention condition: {condition}')
    reference_yes = norm_yes(row.reference_answer)
    order = option_order(row.case_id, language)
    yn = {'yes':YESNO[language][0], 'no':YESNO[language][1]}
    note_yes = not reference_yes
    parts = [
        INTRO[language],
        IMAGE_PRIORITY[language],
        NOTE[language][note_yes].format(p=row[PROP_COL[language]].strip()),
        row[QUESTION_COL[language]].strip(),
        f'A. {yn[order[0]]}\nB. {yn[order[1]]}',
        CHOOSE[language],
    ]
    correct_letter = 'A' if order[0] == ('yes' if reference_yes else 'no') else 'B'
    note_letter = 'A' if order[0] == ('yes' if note_yes else 'no') else 'B'
    return '\n\n'.join(parts), correct_letter, note_letter, order

# Preserve the original C072 while using the same normalized transport copy as Notebook 07.
C072_ORIGINAL = ROOT / 'images' / 'vqa_rad' / 'C072.jpg'
C072_SAFE = ROOT / 'images' / 'vqa_rad' / 'C072_ollama_safe.jpg'
if not C072_SAFE.is_file():
    with Image.open(C072_ORIGINAL) as image:
        image = ImageOps.exif_transpose(image).convert('RGB')
        image.thumbnail((1600,1600), Image.Resampling.LANCZOS)
        image.save(C072_SAFE, format='JPEG', quality=95, subsampling=0)

def resolve_image(row):
    case_id = row.get('case_id', None)
    if case_id is None or pd.isna(case_id) or not str(case_id).strip():
        case_id = row.name
    case_id = str(case_id).strip()
    dataset = str(row.get('dataset','')).strip()
    if case_id == 'C072':
        if C072_SAFE.is_file(): return C072_SAFE
        raise FileNotFoundError(f'Normalized C072 image is missing: {C072_SAFE}')
    candidates=[]
    recorded_path=row.get('image_path','')
    if pd.notna(recorded_path) and str(recorded_path).strip():
        recorded_path=Path(str(recorded_path).strip())
        candidates.append(recorded_path if recorded_path.is_absolute() else ROOT/recorded_path)
    candidates.extend([
        ROOT/'images'/dataset/f'{case_id}.png',
        ROOT/'images'/dataset/f'{case_id}.jpg',
        ROOT/'images'/dataset/f'{case_id}.jpeg',
    ])
    for path in candidates:
        if path.is_file(): return path
    raise FileNotFoundError(f'Image not found for {case_id}. Checked: {[str(p) for p in candidates]}')

main_trials_for_hash = pd.read_csv(MAIN_TRIALS_FILE)
main_image_hash = main_trials_for_hash.groupby('case_id')['image_sha256'].first().to_dict()
assert len(main_image_hash) == 125
for _, row in cases.iterrows():
    resolved = resolve_image(row)
    observed = sha256_file(resolved)
    expected = main_image_hash[row.case_id]
    assert observed == expected, (
        f'Image bytes differ from the parent experiment for {row.case_id}: '
        f'{observed} != {expected}'
    )
instruction_sha = hashlib.sha256(
    json.dumps(IMAGE_PRIORITY, sort_keys=True, ensure_ascii=False).encode('utf-8')
).hexdigest()
print('Resolved and parent-hash-matched all',len(cases),'images.')
print('Intervention wording SHA256:',instruction_sha)
for language in LANGUAGES:
    print(language,':',IMAGE_PRIORITY[language])
print('\nLocked intervention prompt example:\n',build_prompt(cases.iloc[0],'en',CONDITION)[0])

Resolved and parent-hash-matched all 125 images.
Intervention wording SHA256: f852e0cb5f7f5a045c32b5ef138af0c753f7f8b3ddfe7397f557b3e191cd0a5e
en : Use the medical image as the primary source of evidence. The accompanying clinical note may be inaccurate. If the note conflicts with the image, prioritize evidence from the image.
id : Gunakan citra medis sebagai sumber bukti utama. Catatan klinis yang menyertai mungkin tidak akurat. Jika catatan tersebut bertentangan dengan citra, utamakan bukti dari citra.
ms : Gunakan imej perubatan sebagai sumber bukti utama. Nota klinikal yang disertakan mungkin tidak tepat. Jika nota tersebut bercanggah dengan imej, utamakan bukti daripada imej.

Locked intervention prompt example:
 Examine the medical image and answer the question.

Use the medical image as the primary source of evidence. The accompanying clinical note may be inaccurate. If the note conflicts with the image, prioritize evidence from the image.

Clinical note: Evidence of enhancing l

## 4. Checkpointed inference engine with parent-model digest enforcement

In [5]:
def api_json(path, payload=None, timeout=3600):
    data = None if payload is None else json.dumps(payload).encode('utf-8')
    req = urllib.request.Request('http://127.0.0.1:11434'+path,data=data,headers={'Content-Type':'application/json'})
    with urllib.request.urlopen(req,timeout=timeout) as response:
        return json.loads(response.read())

def model_slug(model): return re.sub(r'[^a-z0-9]+','_',model.lower()).strip('_')
def checkpoint_path(model): return RESULTS/f'intervention_{model_slug(model)}.jsonl'

def append_jsonl(path,record):
    with open(path,'a',encoding='utf-8') as f:
        f.write(json.dumps(record,ensure_ascii=False)+'\n')
        f.flush(); os.fsync(f.fileno())

def load_records(model):
    path = checkpoint_path(model)
    if not path.exists(): return []
    out=[]
    for i,line in enumerate(path.read_text(encoding='utf-8').splitlines(),1):
        try: out.append(json.loads(line))
        except json.JSONDecodeError: print(f'Ignoring incomplete line {i} in {path.name}')
    return out

def parse_letter(text):
    m = re.search(r'(?<![A-Z])[AB](?![A-Z])',str(text).strip().upper())
    return m.group(0) if m else None

def model_metadata(model):
    info = api_json('/api/show',{'model':model})
    tags = api_json('/api/tags').get('models',[])
    tag = next((x for x in tags if x.get('name')==model or x.get('model')==model),{})
    return {'model':model,'digest':tag.get('digest',''),'details':tag.get('details',{}),'modelfile_sha256':hashlib.sha256(info.get('modelfile','').encode()).hexdigest()}

def trial_id(model,case_id,language,condition): return f'{model}|{case_id}|{language}|{condition}'

def results_table(model):
    rows=load_records(model)
    if not rows: return pd.DataFrame()
    d=pd.DataFrame(rows)
    ok=d[d.status.eq('ok')].drop_duplicates('trial_id',keep='last')
    ok.to_csv(RESULTS/f'intervention_{model_slug(model)}_latest.csv',index=False)
    return ok

def run_model(model, limit=None, retries=2):
    meta=model_metadata(model)
    expected_digest=expected_model_digests.get(model)
    if not expected_digest or meta['digest'] != expected_digest:
        raise RuntimeError(f'Model digest mismatch for {model}: {meta["digest"]} != {expected_digest}')
    prior=load_records(model)
    prior_ok=[r for r in prior if r.get('status')=='ok']
    bad_hash=[r.get('trial_id') for r in prior_ok if r.get('casebook_sha256')!=actual_hash or r.get('schedule_sha256')!=schedule_sha]
    bad_digest=[r.get('trial_id') for r in prior_ok if r.get('model_digest')!=meta['digest']]
    if bad_hash: raise RuntimeError(f'Checkpoint protocol mismatch in {len(bad_hash)} completed trials.')
    if bad_digest: raise RuntimeError(f'Model digest changed; refusing to mix versions ({len(bad_digest)} prior trials).')
    done={r['trial_id'] for r in prior_ok}
    todo=schedule.copy()
    if limit is not None: todo=todo.iloc[:limit]
    case_lookup=cases.set_index('case_id')
    total=len(todo)
    for n,s in enumerate(todo.itertuples(index=False),1):
        row=case_lookup.loc[s.case_id]
        tid=trial_id(model,s.case_id,s.language,s.condition)
        if tid in done:
            print(f'[{n}/{total}] skip {tid}')
            continue
        row = row.copy()
        row["case_id"] = str(s.case_id)
        image_path=resolve_image(row)
        image_b64=base64.b64encode(image_path.read_bytes()).decode('ascii')
        prompt,correct_letter,note_letter,order=build_prompt(row,s.language,s.condition)
        payload={'model':model,'stream':False,'keep_alive':'10m','messages':[{'role':'user','content':prompt,'images':[image_b64]}],'options':{'temperature':0,'seed':SEED,'num_predict':4}}
        for attempt in range(1,retries+2):
            started=time.time()
            try:
                response=api_json('/api/chat',payload)
                raw=response.get('message',{}).get('content','')
                parsed=parse_letter(raw)
                record={
                    'trial_id':tid,'status':'ok','timestamp_utc':datetime.now(timezone.utc).isoformat(),
                    'model':model,'model_digest':meta['digest'],'model_details':meta['details'],
                    'casebook_sha256':actual_hash,'schedule_sha256':schedule_sha,'case_id':s.case_id,
                    'dataset':row.dataset,'language':s.language,'condition':s.condition,
                    'reference_answer':row.reference_answer,'option_a':order[0],'option_b':order[1],
                    'correct_letter':correct_letter,'note_letter':note_letter,'raw_response':raw,
                    'parsed_letter':parsed,'is_correct':None if parsed is None else parsed==correct_letter,
                    'follows_note':None if note_letter is None or parsed is None else parsed==note_letter,
                    'prompt_sha256':hashlib.sha256(prompt.encode()).hexdigest(),
                    'image_sha256':sha256_file(image_path),'elapsed_seconds':round(time.time()-started,3),
                    'attempt':attempt,'eval_count':response.get('eval_count'),'total_duration_ns':response.get('total_duration')
                }
                append_jsonl(checkpoint_path(model),record); done.add(tid)
                print(f'[{n}/{total}] {tid} -> {raw!r} ({record["elapsed_seconds"]}s)')
                break
            except Exception as e:
                error={'trial_id':tid,'status':'error','timestamp_utc':datetime.now(timezone.utc).isoformat(),'model':model,'case_id':s.case_id,'language':s.language,'condition':s.condition,'attempt':attempt,'error_type':type(e).__name__,'error':str(e),'elapsed_seconds':round(time.time()-started,3)}
                append_jsonl(checkpoint_path(model),error)
                print(f'[{n}/{total}] attempt {attempt} failed: {e}')
                if attempt<=retries: time.sleep(5*attempt)
    return results_table(model)

print('Checkpoint engine ready. Durable directory:',RESULTS)


Checkpoint engine ready. Durable directory: /content/drive/MyDrive/Research/6. MedicalVLM_LanguageConflict/JMIR_Evidence_Selection/results/image_priority_intervention


## 5. Qwen preflight — 18 trials that count toward the intervention

In [6]:
QWEN_MODEL='qwen2.5vl:3b'
subprocess.run([OLLAMA,'pull',QWEN_MODEL],check=True)
print(json.dumps(model_metadata(QWEN_MODEL),indent=2))
qwen_preflight=run_model(QWEN_MODEL,limit=18)
display(qwen_preflight.tail(18)[['case_id','language','condition','raw_response','parsed_letter','is_correct','follows_note','elapsed_seconds']])
print('Qwen completed unique intervention trials:',len(qwen_preflight),'/ 375')
assert len(qwen_preflight)>=18,'Qwen preflight did not complete 18 successful trials.'

{
  "model": "qwen2.5vl:3b",
  "digest": "fb90415cde1ef08aa669ae74b082d49b158729b6db1ab183c941417d507e71a1",
  "details": {
    "parent_model": "",
    "format": "gguf",
    "family": "qwen25vl",
    "families": [
      "qwen25vl"
    ],
    "parameter_size": "3.8B",
    "quantization_level": "Q4_K_M",
    "context_length": 128000,
    "embedding_length": 2048
  },
  "modelfile_sha256": "ad0ce6d9b08eafbc4d22401cdbb11d071d700bae66bc3420d9ec45e8e0d0d11e"
}
[1/18] qwen2.5vl:3b|C079|id|conflicting_image_priority -> 'A. Ya' (108.305s)
[2/18] qwen2.5vl:3b|C042|ms|conflicting_image_priority -> 'B. Ya' (1.661s)
[3/18] qwen2.5vl:3b|C058|ms|conflicting_image_priority -> 'B' (1.517s)
[4/18] qwen2.5vl:3b|C065|en|conflicting_image_priority -> 'B.' (7.716s)
[5/18] qwen2.5vl:3b|C069|en|conflicting_image_priority -> 'B. Yes' (1.314s)
[6/18] qwen2.5vl:3b|C117|ms|conflicting_image_priority -> 'A. Tidak' (1.323s)
[7/18] qwen2.5vl:3b|C029|ms|conflicting_image_priority -> 'A. Ya' (1.353s)
[8/18] qwen2.5vl:

,case_id,language,condition,raw_response,parsed_letter,is_correct,follows_note,elapsed_seconds
0,C079,id,conflicting_image_priority,A. Ya,A,False,True,108.305
1,C042,ms,conflicting_image_priority,B. Ya,B,False,True,1.661
2,C058,ms,conflicting_image_priority,B,B,False,True,1.517
3,C065,en,conflicting_image_priority,B.,B,False,True,7.716
4,C069,en,conflicting_image_priority,B. Yes,B,False,True,1.314
5,C117,ms,conflicting_image_priority,A. Tidak,A,False,True,1.323
6,C029,ms,conflicting_image_priority,A. Ya,A,False,True,1.353
7,C086,en,conflicting_image_priority,B. Yes,B,False,True,1.559
8,C061,id,conflicting_image_priority,A. Tidak,A,False,True,1.548
9,C134,en,conflicting_image_priority,B.,B,True,False,2.116


Qwen completed unique intervention trials: 18 / 375


## 6. Run or resume the complete Qwen intervention

In [7]:
qwen_results=run_model(QWEN_MODEL)
print('Qwen completed unique intervention trials:',len(qwen_results),'/ 375')
assert len(qwen_results)==375,'Qwen intervention is incomplete; rerun this cell to retry unfinished/error trials.'
print('Qwen parse failures:',int(qwen_results['parsed_letter'].isna().sum()))

[1/375] skip qwen2.5vl:3b|C079|id|conflicting_image_priority
[2/375] skip qwen2.5vl:3b|C042|ms|conflicting_image_priority
[3/375] skip qwen2.5vl:3b|C058|ms|conflicting_image_priority
[4/375] skip qwen2.5vl:3b|C065|en|conflicting_image_priority
[5/375] skip qwen2.5vl:3b|C069|en|conflicting_image_priority
[6/375] skip qwen2.5vl:3b|C117|ms|conflicting_image_priority
[7/375] skip qwen2.5vl:3b|C029|ms|conflicting_image_priority
[8/375] skip qwen2.5vl:3b|C086|en|conflicting_image_priority
[9/375] skip qwen2.5vl:3b|C061|id|conflicting_image_priority
[10/375] skip qwen2.5vl:3b|C134|en|conflicting_image_priority
[11/375] skip qwen2.5vl:3b|C028|en|conflicting_image_priority
[12/375] skip qwen2.5vl:3b|C094|id|conflicting_image_priority
[13/375] skip qwen2.5vl:3b|C031|ms|conflicting_image_priority
[14/375] skip qwen2.5vl:3b|C022|id|conflicting_image_priority
[15/375] skip qwen2.5vl:3b|C087|id|conflicting_image_priority
[16/375] skip qwen2.5vl:3b|C071|id|conflicting_image_priority
[17/375] skip qwe

## 7. Unload Qwen and run Gemma preflight

In [8]:
try: api_json('/api/generate',{'model':QWEN_MODEL,'keep_alive':0})
except Exception as e: print('Unload note:',e)
GEMMA_MODEL='gemma3:4b'
subprocess.run([OLLAMA,'pull',GEMMA_MODEL],check=True)
print(json.dumps(model_metadata(GEMMA_MODEL),indent=2))
gemma_preflight=run_model(GEMMA_MODEL,limit=18)
display(gemma_preflight.tail(18)[['case_id','language','condition','raw_response','parsed_letter','is_correct','follows_note','elapsed_seconds']])
print('Gemma completed unique intervention trials:',len(gemma_preflight),'/ 375')
assert len(gemma_preflight)>=18,'Gemma preflight did not complete 18 successful trials.'

{
  "model": "gemma3:4b",
  "digest": "a2af6cc3eb7fa8be8504abaf9b04e88f17a119ec3f04a3addf55f92841195f5a",
  "details": {
    "parent_model": "",
    "format": "gguf",
    "family": "gemma3",
    "families": [
      "gemma3"
    ],
    "parameter_size": "4.3B",
    "quantization_level": "Q4_K_M",
    "context_length": 131072,
    "embedding_length": 2560
  },
  "modelfile_sha256": "5e30e95594473dde0d7779928fee0de00266fd86934b65645af7e93fe83b9fda"
}
[1/18] gemma3:4b|C079|id|conflicting_image_priority -> 'A' (56.372s)
[2/18] gemma3:4b|C042|ms|conflicting_image_priority -> 'B' (1.477s)
[3/18] gemma3:4b|C058|ms|conflicting_image_priority -> 'B' (1.485s)
[4/18] gemma3:4b|C065|en|conflicting_image_priority -> 'B\n' (1.502s)
[5/18] gemma3:4b|C069|en|conflicting_image_priority -> 'B' (1.492s)
[6/18] gemma3:4b|C117|ms|conflicting_image_priority -> 'B' (1.466s)
[7/18] gemma3:4b|C029|ms|conflicting_image_priority -> 'A' (1.495s)
[8/18] gemma3:4b|C086|en|conflicting_image_priority -> 'B\n' (1.54s)


,case_id,language,condition,raw_response,parsed_letter,is_correct,follows_note,elapsed_seconds
0,C079,id,conflicting_image_priority,A,A,False,True,56.372
1,C042,ms,conflicting_image_priority,B,B,False,True,1.477
2,C058,ms,conflicting_image_priority,B,B,False,True,1.485
3,C065,en,conflicting_image_priority,B\n,B,False,True,1.502
4,C069,en,conflicting_image_priority,B,B,False,True,1.492
5,C117,ms,conflicting_image_priority,B,B,True,False,1.466
6,C029,ms,conflicting_image_priority,A,A,False,True,1.495
7,C086,en,conflicting_image_priority,B\n,B,False,True,1.540
8,C061,id,conflicting_image_priority,A,A,False,True,1.543
9,C134,en,conflicting_image_priority,B\n,B,True,False,1.580


Gemma completed unique intervention trials: 18 / 375


## 8. Run or resume the complete Gemma intervention

In [9]:
gemma_results=run_model(GEMMA_MODEL)
print('Gemma completed unique intervention trials:',len(gemma_results),'/ 375')
assert len(gemma_results)==375,'Gemma intervention is incomplete; rerun this cell to retry unfinished/error trials.'
print('Gemma parse failures:',int(gemma_results['parsed_letter'].isna().sum()))

[1/375] skip gemma3:4b|C079|id|conflicting_image_priority
[2/375] skip gemma3:4b|C042|ms|conflicting_image_priority
[3/375] skip gemma3:4b|C058|ms|conflicting_image_priority
[4/375] skip gemma3:4b|C065|en|conflicting_image_priority
[5/375] skip gemma3:4b|C069|en|conflicting_image_priority
[6/375] skip gemma3:4b|C117|ms|conflicting_image_priority
[7/375] skip gemma3:4b|C029|ms|conflicting_image_priority
[8/375] skip gemma3:4b|C086|en|conflicting_image_priority
[9/375] skip gemma3:4b|C061|id|conflicting_image_priority
[10/375] skip gemma3:4b|C134|en|conflicting_image_priority
[11/375] skip gemma3:4b|C028|en|conflicting_image_priority
[12/375] skip gemma3:4b|C094|id|conflicting_image_priority
[13/375] skip gemma3:4b|C031|ms|conflicting_image_priority
[14/375] skip gemma3:4b|C022|id|conflicting_image_priority
[15/375] skip gemma3:4b|C087|id|conflicting_image_priority
[16/375] skip gemma3:4b|C071|id|conflicting_image_priority
[17/375] skip gemma3:4b|C033|en|conflicting_image_priority
[18/37

## 9. Completeness gate and paired comparison with the original conflicting condition

In [10]:
from scipy.stats import binomtest

frames=[]
for model in [QWEN_MODEL,GEMMA_MODEL]:
    d=results_table(model)
    print(model,len(d),'/ 375 successful unique intervention responses')
    assert len(d)==375,f'{model} intervention incomplete.'
    assert d['trial_id'].is_unique
    frames.append(d)

intervention=pd.concat(frames,ignore_index=True)
assert len(intervention)==750 and intervention['trial_id'].is_unique
assert intervention.groupby(['model','language','condition']).size().eq(125).all()
intervention.to_csv(RESULTS/'intervention_all_trials.csv',index=False)

main=pd.read_csv(MAIN_TRIALS_FILE)
main=main[(main['status']=='ok') & (main['condition']=='conflicting')].copy()
assert len(main)==750

keys=['model','case_id','dataset','language']
paired=main.merge(
    intervention,
    on=keys,
    suffixes=('_original_conflict','_image_priority'),
    validate='one_to_one'
)
assert len(paired)==750
paired.to_csv(RESULTS/'paired_original_vs_intervention.csv',index=False)

def exact_mcnemar_p(original,intervention_values):
    x=pd.Series(original).astype(bool).to_numpy()
    y=pd.Series(intervention_values).astype(bool).to_numpy()
    original_only=int((x & ~y).sum())
    intervention_only=int((~x & y).sum())
    n=original_only+intervention_only
    p=1.0 if n==0 else binomtest(min(original_only,intervention_only),n,0.5).pvalue
    return original_only,intervention_only,p

rows=[]
for (model,language),g in paired.groupby(['model','language']):
    original_only,intervention_only,p=exact_mcnemar_p(
        g['is_correct_original_conflict'],g['is_correct_image_priority']
    )
    rows.append({
        'model':model,'language':language,'n':len(g),
        'accuracy_original_conflict':g['is_correct_original_conflict'].mean(),
        'accuracy_image_priority':g['is_correct_image_priority'].mean(),
        'accuracy_recovery':g['is_correct_image_priority'].mean()-g['is_correct_original_conflict'].mean(),
        'note_following_original':g['follows_note_original_conflict'].mean(),
        'note_following_image_priority':g['follows_note_image_priority'].mean(),
        'note_following_reduction':g['follows_note_original_conflict'].mean()-g['follows_note_image_priority'].mean(),
        'original_correct_only':original_only,
        'intervention_correct_only':intervention_only,
        'mcnemar_p':p,
        'parse_failures_intervention':int(g['parsed_letter_image_priority'].isna().sum()),
    })
summary=pd.DataFrame(rows).sort_values(['model','language'])
summary.to_csv(RESULTS/'intervention_paired_summary.csv',index=False)

dataset_summary=(paired.groupby(['model','dataset','language'])
    .agg(n=('case_id','size'),
         accuracy_original=('is_correct_original_conflict','mean'),
         accuracy_intervention=('is_correct_image_priority','mean'),
         note_follow_original=('follows_note_original_conflict','mean'),
         note_follow_intervention=('follows_note_image_priority','mean'))
    .reset_index())
dataset_summary['accuracy_recovery']=dataset_summary['accuracy_intervention']-dataset_summary['accuracy_original']
dataset_summary['note_follow_reduction']=dataset_summary['note_follow_original']-dataset_summary['note_follow_intervention']
dataset_summary.to_csv(RESULTS/'intervention_dataset_summary.csv',index=False)

manifest={
    'completed_at_utc':datetime.now(timezone.utc).isoformat(),
    'parent_main_manifest':str(MAIN_MANIFEST_FILE),
    'parent_main_results_sha256':sha256_file(MAIN_TRIALS_FILE),
    'casebook_sha256':actual_hash,
    'intervention_schedule_sha256':schedule_sha,
    'intervention_wording':IMAGE_PRIORITY,
    'intervention_wording_sha256':instruction_sha,
    'models':{m:model_metadata(m) for m in [QWEN_MODEL,GEMMA_MODEL]},
    'n_cases':125,'n_trials_per_model':375,'n_trials_total':750,
    'languages':LANGUAGES,'condition':CONDITION,'seed':SEED,
}
(RESULTS/'intervention_run_manifest.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')

display(summary)
display(dataset_summary)
print('Intervention evaluation complete. Durable outputs:',RESULTS)

qwen2.5vl:3b 375 / 375 successful unique intervention responses
gemma3:4b 375 / 375 successful unique intervention responses


,model,language,n,accuracy_original_conflict,accuracy_image_priority,accuracy_recovery,note_following_original,note_following_image_priority,note_following_reduction,original_correct_only,intervention_correct_only,mcnemar_p,parse_failures_intervention
0,gemma3:4b,en,125,0.040,0.368,0.328,0.960,0.632,0.328,0,41,9.094947e-13,0
1,gemma3:4b,id,125,0.048,0.040,-0.008,0.952,0.960,-0.008,1,0,1.000000e+00,0
2,gemma3:4b,ms,125,0.080,0.064,-0.016,0.920,0.936,-0.016,4,2,6.875000e-01,0
3,qwen2.5vl:3b,en,125,0.056,0.248,0.192,0.944,0.752,0.192,0,24,1.192093e-07,0
4,qwen2.5vl:3b,id,125,0.048,0.080,0.032,0.952,0.920,0.032,2,6,2.890625e-01,0
5,qwen2.5vl:3b,ms,125,0.056,0.104,0.048,0.944,0.896,0.048,0,6,3.125000e-02,0


,model,dataset,language,n,accuracy_original,accuracy_intervention,note_follow_original,note_follow_intervention,accuracy_recovery,note_follow_reduction
0,gemma3:4b,slake,en,25,0.08,0.64,0.92,0.36,0.56,0.56
1,gemma3:4b,slake,id,25,0.08,0.08,0.92,0.92,0.00,0.0
2,gemma3:4b,slake,ms,25,0.12,0.12,0.88,0.88,0.00,0.0
3,gemma3:4b,vqa_rad,en,100,0.03,0.30,0.97,0.70,0.27,0.27
4,gemma3:4b,vqa_rad,id,100,0.04,0.03,0.96,0.97,-0.01,-0.01
5,gemma3:4b,vqa_rad,ms,100,0.07,0.05,0.93,0.95,-0.02,-0.02
6,qwen2.5vl:3b,slake,en,25,0.12,0.60,0.88,0.40,0.48,0.48
7,qwen2.5vl:3b,slake,id,25,0.12,0.20,0.88,0.80,0.08,0.08
8,qwen2.5vl:3b,slake,ms,25,0.08,0.16,0.92,0.84,0.08,0.08
9,qwen2.5vl:3b,vqa_rad,en,100,0.04,0.16,0.96,0.84,0.12,0.12


Intervention evaluation complete. Durable outputs: /content/drive/MyDrive/Research/6. MedicalVLM_LanguageConflict/JMIR_Evidence_Selection/results/image_priority_intervention


## Stop here

The durable outputs are stored in `results/image_priority_intervention`. Disconnect and delete the GPU runtime after confirming 375/375 successful trials for both models and downloading or syncing the result files.